# Análise Financeira com Python — ClearBank

Notebook do desafio final: leitura e validação do CSV de transações, geração de métricas
mensais, identificação de transações suspeitas, relatório formatado no terminal e
exportação para `relatorio.json`.

Ordem das células: **leitura → validação → datas/métricas → exibição → exportação → execução principal**.


In [1]:
# Bibliotecas nativas apenas (sem pandas) e constantes do desafio.
import csv
import json
from datetime import datetime

LIMITE_SUSPEITO = 10000.00      # valores acima disso são marcados como suspeitos
ARQUIVO_CSV = "transacoes.csv"
ARQUIVO_JSON = "relatorio.json"


## 1. Leitura do CSV (`ler_transacoes`)

In [3]:
def ler_transacoes(caminho):
    # DictReader permite acessar cada coluna pelo nome do cabeçalho.
    try:
        with open(caminho, "r", encoding="utf-8", newline="") as arquivo:
            leitor = csv.DictReader(arquivo)
            return list(leitor)
    except FileNotFoundError:
        print(f"Arquivo não encontrado: {caminho}")
        return []


In [4]:
# Teste rápido: leitura do arquivo
linhas_teste = ler_transacoes(ARQUIVO_CSV)
print(f"Linhas lidas no teste: {len(linhas_teste)}")
assert len(linhas_teste) > 0, "Nenhuma linha foi lida - verifique se transacoes.csv existe"


Linhas lidas no teste: 22


## 2. Validação e limpeza dos dados

In [4]:
def validar_data(texto):
    # strptime lança ValueError para formato errado; AttributeError cobre texto None.
    try:
        return datetime.strptime(texto.strip(), "%Y-%m-%d")
    except (ValueError, AttributeError):
        return None


def validar_valor(texto):
    # float() lança ValueError para texto não numérico; valor precisa ser > 0.
    try:
        valor = float(texto)
        return valor if valor > 0 else None
    except (ValueError, TypeError):
        return None


In [5]:
def validar_transacao(linha):
    # Retorna o registro limpo (com tipos convertidos) ou None se qualquer regra falhar.
    id_bruto = (linha.get("id") or "").strip()
    cliente_id = (linha.get("cliente_id") or "").strip()
    tipo = (linha.get("tipo") or "").strip().lower()

    if not id_bruto.isdigit():
        return None
    if not cliente_id:
        return None
    if tipo not in ("credito", "debito"):
        return None

    data = validar_data(linha.get("data", ""))
    if data is None:
        return None

    valor = validar_valor(linha.get("valor", ""))
    if valor is None:
        return None

    return {
        "id": int(id_bruto),
        "data": data,
        "mes": data.strftime("%Y-%m"),
        "cliente_id": cliente_id,
        "tipo": tipo,
        "valor": valor,
        "descricao": (linha.get("descricao") or "").strip(),
        "categoria": (linha.get("categoria") or "").strip(),
    }


In [6]:
# Teste rápido: validação
assert validar_data("2026-01-05") is not None
assert validar_data("05-01-2026") is None
assert validar_valor("150.50") == 150.50
assert validar_valor("abc") is None
assert validar_valor("-10") is None

exemplo_valido = {"id": "1", "data": "2026-01-05", "cliente_id": "CLI001", "tipo": "credito",
                   "valor": "3500.00", "descricao": "Teste", "categoria": "salario"}
exemplo_invalido = {"id": "", "data": "2026-01-05", "cliente_id": "CLI001", "tipo": "credito",
                     "valor": "3500.00", "descricao": "Teste", "categoria": "salario"}
assert validar_transacao(exemplo_valido) is not None
assert validar_transacao(exemplo_invalido) is None
print("Validações passaram no teste rápido.")


Validações passaram no teste rápido.


## 3. Processamento em lote (`processar_transacoes`)

In [7]:
def processar_transacoes(linhas_brutas):
    # Descarta silenciosamente as linhas inválidas, sem interromper o processamento.
    validas = []
    invalidas = 0
    for linha in linhas_brutas:
        transacao = validar_transacao(linha)
        if transacao is None:
            invalidas += 1
        else:
            validas.append(transacao)

    print("Total de linhas lidas:", len(linhas_brutas))
    print("Linhas válidas:", len(validas))
    print("Linhas inválidas:", invalidas)
    return validas, invalidas


In [8]:
# Teste rápido: processamento em lote
transacoes_validas_teste, invalidas_teste = processar_transacoes(linhas_teste)


Total de linhas lidas: 22
Linhas válidas: 17
Linhas inválidas: 5


## 4. Datas e métricas mensais (`gerar_relatorio`)

In [9]:
def gerar_relatorio(transacoes_validas, total_invalidas):
    resumo_mensal = {}
    suspeitas = []

    for t in transacoes_validas:
        mes = t["mes"]
        bucket = resumo_mensal.setdefault(mes, {
            "quantidade": 0,
            "total_credito": 0.0,
            "total_debito": 0.0,
            "maior_valor": None,
            "menor_valor": None,
        })
        bucket["quantidade"] += 1
        if t["tipo"] == "credito":
            bucket["total_credito"] += t["valor"]
        else:
            bucket["total_debito"] += t["valor"]
        if bucket["maior_valor"] is None or t["valor"] > bucket["maior_valor"]:
            bucket["maior_valor"] = t["valor"]
        if bucket["menor_valor"] is None or t["valor"] < bucket["menor_valor"]:
            bucket["menor_valor"] = t["valor"]

        if t["valor"] > LIMITE_SUSPEITO:
            suspeitas.append(t)

    for dados in resumo_mensal.values():
        dados["saldo"] = dados["total_credito"] - dados["total_debito"]
        dados["media"] = (dados["total_credito"] + dados["total_debito"]) / dados["quantidade"]

    # Período coberto pelos dados: da transação mais antiga à mais recente.
    datas = [t["data"] for t in transacoes_validas]
    periodo = {
        "inicio": min(datas) if datas else None,
        "fim": max(datas) if datas else None,
    }
    dias_periodo = (periodo["fim"] - periodo["inicio"]).days if datas else 0

    return {
        "gerado_em": datetime.now().strftime("%Y-%m-%d"),
        "total_transacoes_validas": len(transacoes_validas),
        "total_transacoes_invalidas": total_invalidas,
        "periodo": periodo,
        "dias_periodo": dias_periodo,
        "resumo_mensal": resumo_mensal,
        "transacoes_suspeitas": suspeitas,
    }


In [10]:
# Teste rápido: geração do relatório
relatorio_teste = gerar_relatorio(transacoes_validas_teste, invalidas_teste)
print(f"Meses no relatório: {sorted(relatorio_teste['resumo_mensal'].keys())}")
print(f"Transações suspeitas: {len(relatorio_teste['transacoes_suspeitas'])}")


Meses no relatório: ['2026-01', '2026-02', '2026-03', '2026-04']
Transações suspeitas: 2


## 5. Exibição formatada no terminal (`exibir_relatorio`)

In [11]:
def formatar_moeda(valor):
    # Converte 1234.5 -> "R$ 1.234,50" (padrão brasileiro).
    texto = f"{valor:,.2f}"
    return "R$ " + texto.replace(",", "X").replace(".", ",").replace("X", ".")


def exibir_relatorio(relatorio):
    print("\n===== RELATÓRIO MENSAL =====")
    for mes in sorted(relatorio["resumo_mensal"]):
        dados = relatorio["resumo_mensal"][mes]
        print(f"\nMês: {mes}")
        print(f"  Transações: {dados['quantidade']}")
        print(f"  Total crédito: {formatar_moeda(dados['total_credito'])}")
        print(f"  Total débito:  {formatar_moeda(dados['total_debito'])}")
        print(f"  Saldo:         {formatar_moeda(dados['saldo'])}")
        print(f"  Média:         {formatar_moeda(dados['media'])}")
        print(f"  Maior valor:   {formatar_moeda(dados['maior_valor'])}")
        print(f"  Menor valor:   {formatar_moeda(dados['menor_valor'])}")

    print("\n===== TRANSAÇÕES SUSPEITAS =====")
    if relatorio["transacoes_suspeitas"]:
        for t in relatorio["transacoes_suspeitas"]:
            data_texto = t["data"].strftime("%Y-%m-%d")
            print(f"ID: {t['id']} | Cliente: {t['cliente_id']} | Data: {data_texto} | "
                  f"Valor: {formatar_moeda(t['valor'])}")
    else:
        print("Nenhuma transação suspeita encontrada.")

    print("\n===== RESUMO GERAL =====")
    inicio = relatorio["periodo"]["inicio"]
    fim = relatorio["periodo"]["fim"]
    if inicio and fim:
        print(f"Período analisado: {inicio.strftime('%Y-%m-%d')} → {fim.strftime('%Y-%m-%d')} "
              f"({relatorio['dias_periodo']} dias)")
    print(f"Total de transações válidas:   {relatorio['total_transacoes_validas']}")
    print(f"Total de transações inválidas: {relatorio['total_transacoes_invalidas']}")


## 6. Exportação do relatório em JSON (`salvar_json`)

In [12]:
def salvar_json(relatorio, caminho):
    # Converte objetos datetime para texto antes de serializar (JSON não tem tipo data nativo).
    dados_serializaveis = {
        "gerado_em": relatorio["gerado_em"],
        "total_transacoes_validas": relatorio["total_transacoes_validas"],
        "total_transacoes_invalidas": relatorio["total_transacoes_invalidas"],
        "periodo": {
            "inicio": relatorio["periodo"]["inicio"].strftime("%Y-%m-%d") if relatorio["periodo"]["inicio"] else None,
            "fim": relatorio["periodo"]["fim"].strftime("%Y-%m-%d") if relatorio["periodo"]["fim"] else None,
        },
        "dias_periodo": relatorio["dias_periodo"],
        "resumo_mensal": {
            mes: {
                "quantidade": dados["quantidade"],
                "total_credito": round(dados["total_credito"], 2),
                "total_debito": round(dados["total_debito"], 2),
                "saldo": round(dados["saldo"], 2),
                "media": round(dados["media"], 2),
                "maior_valor": round(dados["maior_valor"], 2),
                "menor_valor": round(dados["menor_valor"], 2),
            }
            for mes, dados in relatorio["resumo_mensal"].items()
        },
        "transacoes_suspeitas": [
            {
                "id": t["id"],
                "cliente_id": t["cliente_id"],
                "data": t["data"].strftime("%Y-%m-%d"),
                "valor": round(t["valor"], 2),
            }
            for t in relatorio["transacoes_suspeitas"]
        ],
    }
    try:
        with open(caminho, "w", encoding="utf-8") as arquivo:
            json.dump(dados_serializaveis, arquivo, ensure_ascii=False, indent=2)
        print(f"\nRelatório salvo em {caminho}")
    except OSError as erro:
        print(f"Erro ao salvar o relatório: {erro}")


## Célula de Execução Principal

Chama todas as funções em sequência — use-a como validação final do notebook.

In [13]:
def main():
    linhas_brutas = ler_transacoes(ARQUIVO_CSV)
    if not linhas_brutas:
        return
    transacoes_validas, total_invalidas = processar_transacoes(linhas_brutas)
    relatorio = gerar_relatorio(transacoes_validas, total_invalidas)
    exibir_relatorio(relatorio)
    salvar_json(relatorio, ARQUIVO_JSON)


main()


Total de linhas lidas: 22
Linhas válidas: 17
Linhas inválidas: 5

===== RELATÓRIO MENSAL =====

Mês: 2026-01
  Transações: 4
  Total crédito: R$ 18.500,00
  Total débito:  R$ 500,50
  Saldo:         R$ 17.999,50
  Média:         R$ 4.750,12
  Maior valor:   R$ 15.000,00
  Menor valor:   R$ 180,50

Mês: 2026-02
  Transações: 5
  Total crédito: R$ 22.200,00
  Total débito:  R$ 549,90
  Saldo:         R$ 21.650,10
  Média:         R$ 4.549,98
  Maior valor:   R$ 15.500,00
  Menor valor:   R$ 99,90

Mês: 2026-03
  Transações: 6
  Total crédito: R$ 7.000,00
  Total débito:  R$ 1.129,90
  Saldo:         R$ 5.870,10
  Média:         R$ 1.354,98
  Maior valor:   R$ 3.500,00
  Menor valor:   R$ 99,90

Mês: 2026-04
  Transações: 2
  Total crédito: R$ 3.500,00
  Total débito:  R$ 220,00
  Saldo:         R$ 3.280,00
  Média:         R$ 1.860,00
  Maior valor:   R$ 3.500,00
  Menor valor:   R$ 220,00

===== TRANSAÇÕES SUSPEITAS =====
ID: 3 | Cliente: CLI003 | Data: 2026-01-20 | Valor: R$ 15.000,00
